In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
ROUTES_PATH = r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\routes\yandex_izhevsk_28409.gpkg"

In [5]:
routes = gpd.read_file(ROUTES_PATH, layer='yandex_routes')
stops = gpd.read_file(r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\routes\yandex_izhevsk_28409.gpkg", layer='yandex_stops')
isochrones = gpd.read_file(r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\1_data\isochrones_result.gpkg")

In [6]:
stops_1 = stops[['ya_stop_id', 'ya_line_id','ya_route_id']].merge(isochrones[['point_id', 'intermodalnost']]
                                                                  .drop_duplicates(subset='point_id'), left_on='ya_stop_id',right_on='point_id', how='left')

In [7]:
routes = routes[routes['main_route'] == 'true']

In [8]:
stops_1['inter'] = (stops_1['intermodalnost'] > 0)

In [9]:
stops_1['inter'] = stops_1['inter'].fillna(0).astype(int)
stops_1

,ya_stop_id,ya_line_id,ya_route_id,point_id,intermodalnost,inter
0,stop__9884354,"2161483905, 2161483980, 2161483919, 2161483976...","2161483940, 3960623020, 2161484061, 2161484004...",stop__9884354,6,1
1,stop__9884378,"2161483905, 2161483926, 3079437094, 2161483980...","2161483940, 3960623020, 3456270425, 3973084340...",stop__9884378,4,1
2,stop__9884383,"2161483905, 2161483942, 2161483909, 2161483912...","2161483940, 3960623020, 2161484052, 6063079379...",stop__9884383,0,0
3,stop__9884414,"2161483905, 2161483982, 2161483942, 2161483909...","2161483940, 3960623020, 2161484087, 2161484052...",stop__9884414,0,0
4,stop__9884444,"2161483905, 2161483953","2161483940, 3960623020, 2161484059",stop__9884444,2,1
...,...,...,...,...,...,...
683,stop__9884397,2161483903,3765179390,stop__9884397,2,1
684,stop__9884091,"2161483925, 3357137595","2161484017, 3357137945",stop__9884091,0,0
685,stop__9884380,"2161483925, 3357137595","3767018110, 3764809050",stop__9884380,0,0
686,3767017460,2161483925,3767018110,3767017460,0,0


In [10]:
stops_1['ya_route_id'] = stops_1['ya_route_id'].astype(str).str.split(',')
stops_exploded = stops_1.explode('ya_route_id')
stops_exploded['ya_route_id'] = stops_exploded['ya_route_id'].str.strip()

In [11]:
stops_exploded

,ya_stop_id,ya_line_id,ya_route_id,point_id,intermodalnost,inter
0,stop__9884354,"2161483905, 2161483980, 2161483919, 2161483976...",2161483940,stop__9884354,6,1
0,stop__9884354,"2161483905, 2161483980, 2161483919, 2161483976...",3960623020,stop__9884354,6,1
0,stop__9884354,"2161483905, 2161483980, 2161483919, 2161483976...",2161484061,stop__9884354,6,1
0,stop__9884354,"2161483905, 2161483980, 2161483919, 2161483976...",2161484004,stop__9884354,6,1
0,stop__9884354,"2161483905, 2161483980, 2161483919, 2161483976...",2161484006,stop__9884354,6,1
...,...,...,...,...,...,...
685,stop__9884380,"2161483925, 3357137595",3767018110,stop__9884380,0,0
685,stop__9884380,"2161483925, 3357137595",3764809050,stop__9884380,0,0
686,3767017460,2161483925,3767018110,3767017460,0,0
687,3764629460,2161483979,2172348372,3764629460,0,0


In [12]:
result = (
    stops_exploded
    .groupby('ya_route_id')
    .agg(
        total_stops       = ('point_id', 'nunique'),   # всего уникальных остановок
        transfer_stops    = ('inter',       'sum'),       # из них пересадочных
    )
    .reset_index()
)


In [13]:
result

,ya_route_id,total_stops,transfer_stops
0,2161483920,26,26
1,2161483927,33,6
2,2161483936,14,14
3,2161483940,10,3
4,2161483943,15,15
...,...,...,...
148,6541017682,28,28
149,6541028902,7,7
150,6541035752,21,21
151,6755267534,9,4


In [14]:
result['transfer_share'] = (
    result['transfer_stops'] / result['total_stops']
).round(3)

In [15]:
# routes['ya_route_id'] = routes['ya_route_id'].astype(str).str.strip()

result = result.merge(
    routes[['ya_route_id', 'route_name_with_vhc_prefix', 'ya_line_id']],
    on='ya_route_id',
    how='left'
)

# Переупорядочиваем столбцы
result = result[['ya_route_id', 'ya_line_id', 'route_name_with_vhc_prefix', 'total_stops', 'transfer_stops', 'transfer_share']]
result = result.sort_values('transfer_share', ascending=False).reset_index(drop=True)

result

,ya_route_id,ya_line_id,route_name_with_vhc_prefix,total_stops,transfer_stops,transfer_share
0,2161483920,2161483901,tramway_10,26,26,1.0
1,2161483936,2161483904,tramway_11,14,14,1.0
2,2161483943,2161483904,tramway_11,15,15,1.0
3,2161483949,2161483906,tramway_12,26,26,1.0
4,2161483993,2161483947,tramway_8,18,18,1.0
...,...,...,...,...,...,...
148,2161484089,NaN,NaN,17,0,0.0
149,5559412277,NaN,NaN,17,0,0.0
150,4206764043,2161483928,bus_16,24,0,0.0
151,6167904779,NaN,NaN,8,0,0.0


In [16]:
route_to_line = result.groupby('ya_line_id').agg(
    name              = ('route_name_with_vhc_prefix',            'first'),
    num_variants      = ('ya_route_id',        'nunique'),     # сколько вариантов у маршрута
    total_stops       = ('total_stops',      'sum'),        # суммарное кол-во остановок
    transfer_stops    = ('transfer_stops',   'sum'),        # суммарное кол-во пересадочных
    transfer_share_mean = ('transfer_share', 'mean'),       # среднее по вариантам
    transfer_share_min  = ('transfer_share', 'min'),
    transfer_share_max  = ('transfer_share', 'max'),
).reset_index()

# Взвешенная доля (по числу остановок) — обычно корректнее простого среднего
route_to_line['transfer_share_weighted'] = (
    route_to_line['transfer_stops'] / route_to_line['total_stops']
).round(3) * 100

route_to_line = route_to_line.sort_values('transfer_share_weighted', ascending=False).reset_index(drop=True)

route_to_line

,ya_line_id,name,num_variants,total_stops,transfer_stops,transfer_share_mean,transfer_share_min,transfer_share_max,transfer_share_weighted
0,2161483899,tramway_1,2,50,50,1.0000,1.000,1.000,100.0
1,2161483901,tramway_10,2,52,52,1.0000,1.000,1.000,100.0
2,2161483915,tramway_4,2,41,41,1.0000,1.000,1.000,100.0
3,2161483904,tramway_11,2,29,29,1.0000,1.000,1.000,100.0
4,2161483908,tramway_7,2,40,40,1.0000,1.000,1.000,100.0
5,2161483906,tramway_12,2,52,52,1.0000,1.000,1.000,100.0
6,2161483966,tramway_5,2,64,64,1.0000,1.000,1.000,100.0
7,2161483947,tramway_8,2,35,35,1.0000,1.000,1.000,100.0
8,2161483950,tramway_9,2,56,56,1.0000,1.000,1.000,100.0
9,2161483932,tramway_2,2,47,47,1.0000,1.000,1.000,100.0


In [17]:
routes = routes.merge(
    route_to_line[[
        'ya_line_id',
        'total_stops',
        'transfer_stops',
        'transfer_share_weighted',
        'transfer_share_mean',
    ]],
    on='ya_line_id',
    how='left'
)

In [102]:
routes.to_file(ROUTES_PATH, layer="lines_with_transfers", driver="GPKG")

In [18]:
routes

,ya_line_id,ya_route_id,ya_vhc_type,ya_route_name,route_name_with_vhc_prefix,file_name,start_stop_id,end_stop_id,start_stop_name,end_stop_name,...,3_class,8_uds,9_anchor,10_popul,2_start_time,geometry,total_stops,transfer_stops,transfer_share_weighted,transfer_share_mean
0,2161483905,2161483940,bus,11,bus_11,11,stop__9884354,stop__10061013,Ижевск – Собор Александра Невского,Улица Фурманова,...,3.0,32.0,2.0,2.0,5.0,"LINESTRING (9634444.857 6304730.034, 9634418.7...",20,7,35.0,0.3500
1,2161483905,2161483946,bus,11,bus_11,11,stop__9884455,stop__9884352,Улица Фурманова,Ижевск – Собор Александра Невского,...,3.0,32.0,2.0,2.0,5.0,"LINESTRING (9631958.022 6299080.08, 9631958.96...",20,7,35.0,0.3500
2,2161483924,2161483952,bus,12,bus_12,12,stop__9884357,stop__9884310,Станкострой,Автозавод - Разворотное кольцо,...,3.0,31.0,2.0,14.0,4.0,"LINESTRING (9633034.046 6304670.72, 9633036.20...",42,15,35.7,0.3570
3,2161483924,2161483955,bus,12,bus_12,12,stop__9884309,stop__9884357,Автозавод – Разворотное кольцо,Станкострой,...,3.0,31.0,2.0,14.0,4.0,"LINESTRING (9640668.133 6309216.425, 9640637.0...",42,15,35.7,0.3570
4,2161483926,2161484020,bus,15,bus_15,15,stop__9884471,stop__9884357,Металлобаза,Станкострой,...,3.0,57.0,2.0,4.0,5.0,"LINESTRING (9639115.503 6303107.303, 9639116.1...",34,9,26.5,0.2645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,3357137595,3357137945,trolleybus,7,trolleybus_7,tr7,3355529325,stop__9884082,Центр,Улица Труда,...,3.0,33.0,2.0,19.0,5.0,"LINESTRING (9634525.711 6304722.608, 9634516.3...",46,12,26.1,0.2615
90,3357137595,3764809050,trolleybus,7,trolleybus_7,tr7,stop__9884380,stop__9884351,Улица Труда,Центр,...,3.0,33.0,2.0,19.0,5.0,"LINESTRING (9639868.776 6306797.12, 9639862.82...",46,12,26.1,0.2615
91,2161483951,2161484058,trolleybus,9,trolleybus_9,tr9,stop__9884404,stop__9884109,Посёлок Машиностроителей,Городок Металлургов,...,3.0,31.0,2.0,18.0,5.0,"LINESTRING (9628966.413 6303341.049, 9628995.3...",46,22,47.8,0.4780
92,2161483942,6063076059,bus,36,bus_36,36,stop__9884229,stop__9884311,Микрорайон Нагорный,Завод пластмасс,...,3.0,28.0,2.0,29.0,5.0,"LINESTRING (9632982.617 6298689.604, 9632992.3...",82,32,39.0,0.3925
